# 01 · 美国交通事故数据切片
从事故记录走到“某个片区下一周是否有记录”。请从头运行。原始数据留在 `data/`，输出的样本与中文报告使用项目相对路径。环境为 Python (7308-Final-project)，依赖 pandas、numpy、pyarrow 和 Jupyter。

## 1　先看七个文件装了什么
只读文件元数据，核对表结构、行数和磁盘大小。文件名里的 train 表示发布分区；项目应另行按时间切分。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

project = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                if (p / 'data').is_dir() and (p / 'requirements.txt').is_file()), None)
if project is None:
    raise RuntimeError('请从项目根目录或 notebooks 目录启动 notebook')
files = sorted((project / 'data').glob('train-*-of-*.parquet'))
assert len(files) == 7, '本次发布应包含七个分片'
shards = [pq.ParquetFile(p) for p in files]
schema = shards[0].schema_arrow
assert all(s.schema_arrow.equals(schema) for s in shards)
manifest = pd.DataFrame([{'文件': p.name, '行数': s.metadata.num_rows,
                          '列数': len(schema), '大小_MB': round(p.stat().st_size / 1e6, 2)}
                         for p, s in zip(files, shards)])
print(sys.executable, '| pyarrow', pa.__version__)
display(manifest)
print('总记录数', manifest['行数'].sum())

## 2　取一个可重现的小切片
每个文件沿行组顺序分为四段，每段随机选一个足够大的行组，再等概率抽取 250 条。固定种子。这个设计方便浏览不同位置的记录，不能用来估计全国事故率，也不能拿它补出所有无事故周。

In [ ]:
rng = np.random.default_rng(7308)
parts = []
for shard in shards:
    groups = [i for i in range(shard.num_row_groups) if shard.metadata.row_group(i).num_rows >= 250]
    for segment in np.array_split(groups, 4):
        group = shard.read_row_group(int(rng.choice(segment)))
        indices = np.sort(rng.choice(group.num_rows, 250, replace=False))
        parts.append(group.take(indices))
sample = pa.concat_tables(parts).to_pandas()
out = project / 'outputs' / 'us_accidents'
out.mkdir(parents=True, exist_ok=True)
sample.to_parquet(out / 'sample_7000.parquet', index=False)
display(sample[['ID','Start_Time','City','State','Start_Lat','Start_Lng','Severity']].head())

## 3　逐列读表头
表中类型来自原 Parquet。缺失只统计空值，示例取切片中第一个非空值。布尔值 False 表示该条记录未标记这一设施。

In [ ]:
# 以下名称与原表顺序一一对应。校验列名以防数据源变更时发生错配。
groups = {
    '记录与影响': {'ID':'记录编号','Source':'交通信息来源代码','Severity':'对交通的影响等级，1 至 4 逐渐加重','Distance(mi)':'受影响路段长度（英里）','Description':'事件文字通报'},
    '发生时空': {'Start_Time':'事件开始时间（当地时间）','End_Time':'记录的交通影响结束时间','Start_Lat':'起始纬度','Start_Lng':'起始经度','End_Lat':'结束位置纬度','End_Lng':'结束位置经度'},
    '地址': {'Street':'街道','City':'城市','County':'县','State':'州缩写','Zipcode':'邮编','Country':'国家','Timezone':'时区'},
    '天气': {'Airport_Code':'关联天气站的机场代码','Weather_Timestamp':'天气观测时间','Temperature(F)':'气温（华氏度）','Wind_Chill(F)':'风寒温度（华氏度）','Humidity(%)':'相对湿度（%）','Pressure(in)':'气压（英寸汞柱）','Visibility(mi)':'能见度（英里）','Wind_Direction':'风向','Wind_Speed(mph)':'风速（英里/小时）','Precipitation(in)':'降水量（英寸）','Weather_Condition':'天气描述'},
    '附近设施标记': {'Amenity':'生活服务设施','Bump':'减速凸起','Crossing':'人行横道等过街设施','Give_Way':'让行标志','Junction':'道路交汇处','No_Exit':'无出口标记','Railway':'铁路','Roundabout':'环岛','Station':'交通站点','Stop':'停车让行标志','Traffic_Calming':'交通减速设施','Traffic_Signal':'交通信号灯','Turning_Loop':'回转环路'},
    '光照': {'Sunrise_Sunset':'以日出日落界定昼夜','Civil_Twilight':'按民用曙暮光界定昼夜','Nautical_Twilight':'按航海曙暮光界定昼夜','Astronomical_Twilight':'按天文曙暮光界定昼夜'}
}
lookup = {col: (category, meaning) for category, fields in groups.items() for col, meaning in fields.items()}
assert set(lookup) == set(sample.columns)
rows = []
for field in schema:
    col = sample[field.name]
    example = str(col.dropna().iloc[0])[:85] if col.notna().any() else '—'
    rows.append([field.name, *lookup[field.name], str(field.type), int(col.isna().sum()), example])
profile = pd.DataFrame(rows, columns=['字段','分组','怎么读','原类型','切片空值数','样例'])
with pd.option_context('display.max_rows', None):
    display(profile)

## 4　切片中实际看到了什么
先查经纬度、时间可用性，再看严重度与道路设施标记。百分比的分母是切片行数。

In [ ]:
dates = pd.to_datetime(sample.Start_Time, format='mixed', errors='coerce')
coords_ok = sample.Start_Lat.between(-90,90) & sample.Start_Lng.between(-180,180)
severity = sample.Severity.value_counts().sort_index().rename_axis('交通影响等级').reset_index(name='条数')
severity['占切片'] = severity['条数'].map(lambda n: f'{n / len(sample):.1%}')
roads = sample[list(groups['附近设施标记'])].sum().sort_values(ascending=False)
print('起始时间可解析', dates.notna().sum(), '经纬度在合法范围内', coords_ok.sum())
display(severity, roads.to_frame('标记为真的条数'))

## 5　从一条事故记录到一个预测样本
以洛杉矶市区附近一个固定经纬度方框演示。逐个读取七个分片的坐标和时间，数出方框中连续 12 周的记录。仅用这个小方框解释流程，正式分析再选城市、用投影坐标建等距网格。表中空的历史窗口表示可用周数不足。

In [ ]:
# 坐标边界左闭右开。全分片扫描，不从 7000 条切片数周记录。
west, east, south, north = -118.26, -118.25, 34.03, 34.04
weeks = pd.date_range('2022-05-02', periods=12, freq='W-MON')
end = weeks[-1] + pd.Timedelta(weeks=1)
counts = pd.Series(0, index=weeks, dtype='int64')
invalid_start_count = 0
for shard in shards:
    for batch in shard.iter_batches(batch_size=65536, columns=['Start_Time','Start_Lat','Start_Lng']):
        frame = batch.to_pandas()
        inside = (frame.Start_Lng.ge(west) & frame.Start_Lng.lt(east)
                  & frame.Start_Lat.ge(south) & frame.Start_Lat.lt(north))
        t = pd.to_datetime(frame.loc[inside,'Start_Time'], format='mixed', errors='coerce')
        invalid_start_count += int(t.isna().sum())
        t = t[t.ge(weeks[0]) & t.lt(end)]
        monday = t.dt.normalize() - pd.to_timedelta(t.dt.weekday, unit='D')
        counts += monday.value_counts().reindex(weeks, fill_value=0)
weekly = counts.rename('本周记录数').to_frame()
weekly['周一已知的过去4周记录总数'] = counts.shift(1).rolling(4).sum().astype('Int64')
weekly['周末揭晓的标签'] = counts.gt(0).astype(int)
weekly.index.name = '周一'
print('方框内开始时间无法解析、已跳过的记录数', invalid_start_count)
display(weekly)

## 6　生成小报告
将本次运行的统计值、字段字典与连续周实例写入 Markdown。

In [ ]:
def markdown_table(df):
    def cell(value):
        if pd.isna(value):
            return '—'
        return str(value).replace('|', '&#124;').replace('\n', ' ')
    return '\n'.join(['| ' + ' | '.join(df.columns) + ' |',
                      '| ' + ' | '.join(['---'] * len(df.columns)) + ' |',
                      *['| ' + ' | '.join(map(cell, row)) + ' |' for row in df.itertuples(index=False, name=None)]])

first = sample.iloc[0]
week_table = weekly.iloc[4:].reset_index()
week_table['周一'] = week_table['周一'].dt.strftime('%Y-%m-%d')
ex = week_table.iloc[0]
missing = sample.isna().sum().sort_values(ascending=False).head(3)
report = f"""# US Accidents · 从一条事故记录到下一周的预测

**我们想知道，每周一看过去的记录，能不能提前找出接下来七天较可能出现事故记录的片区？** 这份小报告先检查手头的数据能提供什么，再用一个真实片区演示如何出题和对答案。

## 1　打开数据后看到了什么

本地 7 个 Parquet 分片共有 **{manifest['行数'].sum():,} 条记录、{len(schema)} 列**，占用 {sum(p.stat().st_size for p in files)/1e9:.2f} GB。各分片表头一致。发布时这七个文件都归在 train 分区，做预测时仍要自己按时间切分。

从每个分片的四个不同位置各抽 250 条，得到 **{len(sample):,} 条**（固定种子 7308）。以下字段统计只描述这一切片。可直接打开[样本 Parquet](../outputs/us_accidents/sample_7000.parquet)或[已运行的 notebook](../notebooks/01_us_accidents_slice.ipynb)复查。

举一条切片中的真实记录。编号 **{first.ID}** 在 {first.Start_Time} 出现在 {first.City}（{first.State}），坐标为纬度 {first.Start_Lat:.5f}、经度 {first.Start_Lng:.5f}。它给了地图上一个点，也给了日历上一个时刻。起始时间可解析的有 {dates.notna().sum():,} 条，起点经纬度都在合法范围内的有 {coords_ok.sum():,} 条。

## 2　哪些信息对预测有用

| 数据告诉我们什么 | 怎样用 |
| --- | --- |
| 事故何时、何地被记录 | 用 Start_Time、Start_Lat、Start_Lng 把点归到片区和周。ID 用于核对记录。第一版只需这些字段。 |
| 交通受到多大影响 | Severity 表示交通影响，Description 描述已经出现的事件。预测未来一周时，当周事故的这些信息在周一还看不到。 |
| 事发时的天气、光照和附近设施 | 用于描述已有事故。若以后想预测所有片区，应为没有事故的片区也准备同口径的地图设施或预测时已经可得的天气资料。 |
| 某条记录缺什么 | 切片中空值最多的三列为 { '、'.join(f'{k}（{v:,} 条）' for k,v in missing.items()) }。使用事故起点建图时，仍可保留缺少终点的记录。 |

交通影响等级在切片中分布如下。这里的等级衡量道路交通受到的影响，不表示伤亡程度。

{markdown_table(severity)}

道路设施中，切片最常见的三个标记为 { '、'.join(f'{k}（{int(v):,} 条）' for k,v in roads.head(3).items()) }。这些是事故记录里的标记次数。要比较有无信号灯路段的事故风险，还需要知道两类路段各有多少，以及有多少车辆经过。

## 3　用洛杉矶的一小块地图演示

选取经度 [{west}, {east})、纬度 [{south}, {north}) 的小方框，按当地开始时间扫描全部七个分片，只数 2022 年 5 月 2 日起的连续 12 周。前四周用来积累历史。以下每行都是**同一个地方在不同的星期一**。
本次跳过方框内 {invalid_start_count} 条无法解析开始时间的记录。

{markdown_table(week_table)}

读第一行。在 {ex['周一']} 周一早上，我们知道这个方框在此前四周共有 **{ex.iloc[2]} 条**记录。等这一周过完，数据里实际记了 **{ex.iloc[1]} 条**，所以这一周的答案是 **{ex.iloc[3]}**。规则是至少一条记 1，零条记 0。这里的零指数据中没有记录。若整座城市出现采集断档，要把那段时间单独处理。这个方框用于教学，尚未选定正式研究城市或网格大小。

## 4　接下来小组具体做什么

1. **做一张可以训练的表。** 选一个记录较连续的城市，固定城市内的网格，给每个网格列出每一周。用之前几周本格和邻格的记录做输入，用接下来一周是否有记录做答案。用城市完整记录制表，避免把抽样遗漏当成零。
2. **先算一个简单答案。** 假设过去十周有三周出现过记录，最简单的预测就是 30%。这个办法构成其他模型要比较的基线。
3. **检验一个地理问题。** 用一个小型卷积神经网络（CNN）读取过去几周的网格图。分别让它只看本格、也能看周边八格，比较邻近地区的信息是否有帮助。结果可能因城市和网格尺度而异。
4. **用后来的周考试。** 较早的周训练，中间的周选模型和分类阈值，最后的周逐周预测。用 F1 衡量命中、误报和漏报的平衡，也看每周排名前 10 的网格覆盖了多少实际事故记录。展示时用底色画出预测概率，再把实际事故点叠上去。

本次实作完成了原表检查、切片和单个方框的真实周标签演示。正式的全城训练、模型概率和测试分数属于后续实验。

## 附录　46 列逐列速查

类型取自本地 Parquet。示例值来自切片，空值数以 {len(sample):,} 条为分母。设施列的 True 表示该条记录标有这种附近设施。表头保留英文，便于和代码对照。

{markdown_table(profile)}

数据说明见 [Hugging Face 数据卡](https://huggingface.co/datasets/yuvidhepe/us-accidents-updated) 和[原作者字段说明](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents/metadata)。
"""
dest = project / 'reports' / 'US_Accidents_切片.md'
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(report, encoding='utf-8')
print(dest.name)